# EXP-026 — Direct VLM sentiment learning (research only)

Use the existing `exp023_handoff.zip` in MyDrive. Select a **Colab L4**, review Configuration, explicitly authorize the **pilot**, then **Runtime → Run all**. The notebook installs a separate pinned Python3.12.10 environment. Images and model weights stay in `/content`; vision tokens, frozen hidden states, completed training epochs/models and logs persist under a new `MyDrive/exp026` directory.

**F0:** frozen NF4 Qwen decoder states → language-conditioned sentiment head. **L0:** identical inputs/head plus decoder Q/V QLoRA, with the vision tower frozen. **F0_image** is the neutral-prompt diagnostic. No captions, generated answers, CLIP fusion, Development labels or confirmation annotations. Competition eligibility remains unconfirmed; E50+L stays production.

The default pilot uses192 inner-training and48 inner-stopping paintings, two epochs, and reports finite losses, actual gradients/updates, throughput, peak VRAM and a conservative one-fold estimate. It does **not** measure held-out performance. After reviewing the pilot, a separate explicit `STAGE="one-fold"` authorization can resume the exact completed pilot and automatically perform nested stopping/fresh refit and evaluate original P2 fold1 if the measured budget fits. **Full five-fold CV is not offered or automatically launched.** Current local authorization does not authorize this Colab run.

`MAX_HOURS` includes setup/downloads and reserves180 seconds for export. It is a maximum, not a target. Each completed image/epoch/model is saved directly to Drive. Abrupt VM loss can lose the current unfinished epoch; exact compatible completed boundaries resume. Changed source/config/runtime/inputs require a fresh run identity, never reuse old scientific models. Outputs.zip includes logs/checkpoints and is read-back verified before the browser download request and runtime release. Cache files persist separately. API requests do not prove local browser arrival or physical GPU deallocation.


In [ ]:
# OWNER CONFIGURATION — future Colab GPU execution requires fresh authorization.
AUTHORIZE_EXP026 = False
EXECUTION_SCOPE = "RESEARCH_ONLY_CV"
STAGE = "pilot"  # "pilot", or explicitly authorize "one-fold" (pilot then fold 1).
MAX_HOURS = 2.0  # Maximum since Run all, including setup; not a target duration.
INPUT_ZIP = "/content/drive/MyDrive/exp023_handoff.zip"
PERSISTENT_ROOT = "/content/drive/MyDrive/exp026"
LOCAL_ROOT = "/content/exp026"
AUTO_BROWSER_DOWNLOAD = True
AUTO_RELEASE_COLAB = True
SOURCE_REVISION = "a12491e9d5f348e0f44616ea865f05f439f9bbe5"
ZIP_SHA256 = "2db4a53679940c7eb63c7c8272b91be8ec0618bb6c04a69ebaa15909e20f64e0"


In [ ]:
import csv, hashlib, json, math, os, subprocess, sys, time, traceback, zipfile
from pathlib import Path


def stage_command(python, source, inputs, persistent, local, settings, remaining):
    if settings["STAGE"] not in ("pilot", "one-fold"):
        raise ValueError("Choose pilot or one-fold. Full CV is not offered.")
    return [python, "-u", str(source/"scripts/run_exp026.py"), "--stage", settings["STAGE"],
            "--inputs", str(inputs), "--cache", str(persistent/"features"),
            "--run-dir", str(persistent/("run_"+settings["SOURCE_REVISION"][:12])),
            "--model-cache", str(local/"model"), "--max-seconds", str(remaining), "--authorize-gpu"]


def run_notebook(settings, drive, files, runtime):
    started = time.monotonic()
    local, persistent = Path(settings["LOCAL_ROOT"]), Path(settings["PERSISTENT_ROOT"])
    attempt = persistent/"attempts"/str(time.time_ns())
    result = {"status":"STARTING", "research_only":True, "production":"E50+L unchanged",
              "browser_arrival":"UNVERIFIED", "physical_GPU_deallocation":"UNVERIFIED"}
    mounted = False
    run_directory = persistent/("run_"+settings["SOURCE_REVISION"][:12])
    source = local/"source"

    def remaining():
        value = float(settings["MAX_HOURS"])*3600 - (time.monotonic()-started) - 180
        if value <= 30: raise TimeoutError("Allocation exhausted; completed persistent boundaries remain available")
        return value

    def run(command, name, env=None, timeout=None):
        print("Running", name, flush=True)
        log = attempt/(name+".log")
        with log.open("w") as handle:
            process = subprocess.run(command, stdout=handle, stderr=subprocess.STDOUT,
                                     timeout=remaining() if timeout is None else timeout, env=env)
        print("\n".join(log.read_text(errors="replace").splitlines()[-20:]))
        if process.returncode:
            raise RuntimeError(f"{name} stopped (exit {process.returncode}). Inspect {log}; do not change scientific settings to force PASS.")

    def checksum(path):
        with path.open("rb") as handle: return hashlib.file_digest(handle,"sha256").hexdigest()

    try:
        drive.mount("/content/drive"); mounted = True
        attempt.mkdir(parents=True,exist_ok=False); local.mkdir(parents=True,exist_ok=True)
        if not settings["AUTHORIZE_EXP026"]:
            raise PermissionError("No GPU work launched. Explicitly authorize this pilot/one-fold allocation in Configuration before Run all.")
        if settings["EXECUTION_SCOPE"]!="RESEARCH_ONLY_CV":
            raise ValueError("EXP-026 is research-only; competition eligibility remains unresolved.")
        if settings["STAGE"] not in ("pilot","one-fold"):
            raise ValueError("Choose pilot or one-fold; no automatic full CV.")
        if not math.isfinite(float(settings["MAX_HOURS"])) or float(settings["MAX_HOURS"])<=0:
            raise ValueError("MAX_HOURS must be positive and finite")
        if checksum(Path(settings["INPUT_ZIP"]))!=settings["ZIP_SHA256"]:
            raise ValueError("Input ZIP SHA-256 differs. Use the unchanged TASK-025C exp023_handoff.zip.")
        if not source.exists():
            run(["git","clone","--no-checkout","https://github.com/bqa100507-spec/SemEval-2027-Task-12---ArtEmotion-28-Subtask-3---Image-Only-Cross-Cultural-Sentiment-Prediction.git",str(source)],"clone")
            run(["git","-C",str(source),"checkout","--detach",settings["SOURCE_REVISION"]],"checkout")
        if subprocess.check_output(["git","-C",str(source),"rev-parse","HEAD"],text=True).strip()!=settings["SOURCE_REVISION"]:
            raise ValueError("Existing source differs. Choose a new LOCAL_ROOT; do not overwrite historical runs.")
        if subprocess.check_output(["git","-C",str(source),"status","--porcelain","--untracked-files=no"],text=True).strip():
            raise ValueError("Pinned tracked source was modified; restore a clean checkout in a fresh LOCAL_ROOT.")
        config = json.loads((source/"configs/exp026.json").read_text())
        inputs = local/"inputs"
        with zipfile.ZipFile(settings["INPUT_ZIP"]) as archive:
            protocol = archive.read("training/protocol.csv")
            if hashlib.sha256(protocol).hexdigest()!=config["input_sha256"]["training/protocol.csv"]:
                raise ValueError("Protocol checksum differs")
            rows = [row for row in csv.DictReader(protocol.decode().splitlines()) if row["role"]=="development"]
            if len(rows)!=1335: raise ValueError("Expected exactly1335 P2 original training paintings")
            targets = {**config["input_sha256"],**{"artwork/images/"+row["image"]:row["image_sha256"] for row in rows}}
            # Extract only training pairs/protocol/reference and1335 images; no captions or sealed labels.
            for name, expected in targets.items():
                target = (inputs/name).resolve()
                if not target.is_relative_to(inputs.resolve()): raise ValueError("Unsafe ZIP path")
                if target.exists():
                    if checksum(target)!=expected: raise ValueError("Existing local input changed: "+name)
                    continue
                raw = archive.read(name)
                if hashlib.sha256(raw).hexdigest()!=expected: raise ValueError("Input checksum differs: "+name)
                target.parent.mkdir(parents=True,exist_ok=True)
                temporary = target.with_suffix(target.suffix+".pending")
                temporary.write_bytes(raw); temporary.replace(target)
        receipt = {"source_revision":settings["SOURCE_REVISION"], "input_zip_sha256":settings["ZIP_SHA256"],
                   "input_sha256":config["input_sha256"], "stage":settings["STAGE"],
                   "source_sha256":{name:checksum(source/name) for name in ("configs/exp026.json","requirements-exp026.lock.txt",
                     "scripts/run_exp026.py","src/st3/vlm_supervised.py","src/st3/vlm_supervised_training.py")}}
        (attempt/"identity.json").write_text(json.dumps(receipt,indent=2))
        env = dict(os.environ, UV_PYTHON_INSTALL_DIR=str(local/"python"), UV_CACHE_DIR=str(local/"uv_cache"), HF_HUB_DISABLE_XET="1")
        run([sys.executable,"-m","pip","install","uv==0.7.13"],"uv",env)
        run([sys.executable,"-m","uv","python","install","3.12.10"],"python",env)
        if not (local/"venv").exists():
            run([sys.executable,"-m","uv","venv","--python","3.12.10",str(local/"venv")],"venv",env)
        python = str(local/"venv/bin/python")
        run([sys.executable,"-m","uv","pip","sync","--python",python,"--extra-index-url","https://download.pytorch.org/whl/cu126",
             "--index-strategy","unsafe-best-match",str(source/"requirements-exp026.lock.txt")],"dependencies",env)
        run([sys.executable,"-m","uv","pip","check","--python",python],"dependency_check",env)
        run([python,"-c","import torch; assert torch.cuda.is_available(), 'Select a CUDA GPU'; print(torch.cuda.get_device_name()); assert 'L4' in torch.cuda.get_device_name(), 'Select a Colab L4'; assert torch.cuda.mem_get_info()[0]>10*1024**3, 'Need at least10GiB free VRAM'"],"GPU_check",env)
        # Worker has its own shorter hard deadline, so its child exits before this supervisor.
        worker_seconds = remaining()-30
        run(stage_command(python,source,inputs,persistent,local,settings,worker_seconds),"experiment",env,timeout=worker_seconds+20)
        report = json.loads((run_directory/(settings["STAGE"]+"_report.json")).read_text())
        result.update(report)
        print("Measured report:",json.dumps(report,indent=2))
        table = run_directory/"one_fold_metrics.csv"
        if report.get("scientific_fold_completed") and table.exists():print(table.read_text())
        else: print("Pilot only: no held-out scientific performance measured. Review actual throughput/forecast before authorizing one-fold.")
    except Exception as error:
        result.update(status="EXECUTION_STOP",error=f"{type(error).__name__}: {error}",traceback=traceback.format_exc())
    finally:
        try:
            if mounted:
                attempt.mkdir(parents=True,exist_ok=True)
                (attempt/"status.json").write_text(json.dumps(result,indent=2))
                output = attempt/"outputs.zip"
                manifest = {}
                with zipfile.ZipFile(output,"w",zipfile.ZIP_DEFLATED) as archive:
                    for directory,prefix in ((attempt,"attempt"),(run_directory,"run")):
                        if not directory.exists():continue
                        for path in sorted(directory.rglob("*")):
                            if not path.is_file() or path==output or ".pending" in path.name:continue
                            name = prefix+"/"+path.relative_to(directory).as_posix()
                            manifest[name]=checksum(path);archive.write(path,name)
                    archive.writestr("manifest_sha256.json",json.dumps(manifest,indent=2))
                with zipfile.ZipFile(output) as archive:
                    for name, expected in manifest.items():
                        if hashlib.sha256(archive.read(name)).hexdigest()!=expected:
                            raise IOError("Persistent output read-back failed: "+name)
                (attempt/"outputs.sha256").write_text(checksum(output)+"  outputs.zip\n")
                result["persistent_readback"]="PASS"
                result["feature_cache"]="Completed image caches persist separately under "+str(persistent/"features")
                if settings["AUTO_BROWSER_DOWNLOAD"]:
                    result["download_request"]="REQUESTED"
                    try:files.download(str(output));result["download_request"]="API_RETURNED"
                    except Exception as error:result["download_request"]="FAILED: "+str(error)
        except Exception as error:result["export_error"]=str(error)
        finally:
            result["runtime_release"]="REQUESTED_NOT_CONFIRMED" if settings["AUTO_RELEASE_COLAB"] else "DISABLED"
            if mounted:
                try:(attempt/"final_report.json").write_text(json.dumps(result,indent=2))
                except Exception:pass
            print(json.dumps(result,indent=2))
            if settings["AUTO_RELEASE_COLAB"]:
                try:runtime.unassign()
                except Exception as error:print("Runtime release request failed:",error)
    return result


In [ ]:
from google.colab import drive, files, runtime
settings = {key: globals()[key] for key in ("AUTHORIZE_EXP026", "EXECUTION_SCOPE", "STAGE", "MAX_HOURS", "INPUT_ZIP", "PERSISTENT_ROOT", "LOCAL_ROOT", "AUTO_BROWSER_DOWNLOAD", "AUTO_RELEASE_COLAB", "SOURCE_REVISION", "ZIP_SHA256")}
result = run_notebook(settings, drive, files, runtime)
